# DS-GA 1011 A4 on Kaggle

**Before running:** Settings (right panel) → *Accelerator* = **GPU** (P100 or T4) · *Internet* = **On** · the `nlp_a4` code dataset added under *Input*.

1. Edit `JOBS` below.
2. Quick checks → run cells interactively. Long runs → **Save Version → Save & Run All (Commit)**, which keeps going after you close the tab (max 12 h).
3. Results land in `/kaggle/working/a4_outputs` → notebook's **Output** tab.

In [ ]:
# What to run, in order. Each entry: "<part> <stage> [extra flags]"
#   part1: debug | full | transform | augment | augeval | all
#   part2: debug | baseline [flags...] | scratch [flags...]
# An entry can also be a LIST of jobs: they run at the same time, job i on GPU i (Kaggle T4 x2).
JOBS = [
    ["part2 scratch --experiment_name ec2_lr3e-4",
     "part2 scratch --learning_rate 1e-4 --experiment_name ec2_lr1e-4"],
]


In [ ]:
!nvidia-smi -L
import glob, os
hits = glob.glob('/kaggle/input/**/kaggle/run_kaggle.sh', recursive=True)
assert hits, "Code dataset not found: add your nlp_a4 dataset under Input (right panel)"
RUNNER = hits[0]
print("Using", RUNNER)

In [ ]:
import subprocess, sys, re, time, os, shutil, threading
# Progress bars (tqdm) update many times a second, which floods Kaggle's output view until it stops
# rendering. Show a progress line at most once every 60 s (per job); everything else prints normally.
# The full, unfiltered output is in /kaggle/working/a4_outputs/logs/.
PROGRESS = re.compile(r"\d+%\||it/s\]|s/it\]|examples/s\]|B/s\]")

def start(job, gpu=None):
    env = dict(os.environ)
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    return subprocess.Popen(["bash", RUNNER, *job.split()], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)

def stream(p, tag):
    last = 0.0
    for line in p.stdout:
        if PROGRESS.search(line) and "100%" not in line:
            if time.time() - last < 60:
                continue
            last = time.time()
        print(f"{tag}{line.strip()}", flush=True)

for entry in JOBS:
    group = entry if isinstance(entry, list) else [entry]
    print(f"\n{'#'*80}\n# " + "\n# ".join(group) + f"\n{'#'*80}", flush=True)
    procs = [(job, start(job, gpu=i if len(group) > 1 else None)) for i, job in enumerate(group)]
    threads = [threading.Thread(target=stream, args=(p, f"[gpu{i}] " if len(group) > 1 else ""))
               for i, (_, p) in enumerate(procs)]
    for t in threads: t.start()
    for t in threads: t.join()
    for job, p in procs:
        if p.wait() != 0:
            print(f"!! {job} failed (exit {p.returncode}). Full log in /kaggle/working/a4_outputs/logs/")

# Keep the trained BERT models (~430 MB each) so later sessions can re-evaluate without retraining
for d in ["out", "out_augmented"]:
    src = f"/tmp/nlp_a4/part1/{d}"
    if os.path.isdir(src):
        shutil.copytree(src, f"/kaggle/working/a4_outputs/part1/{d}", dirs_exist_ok=True)
        print("kept BERT model:", d)


In [ ]:
!find /kaggle/working/a4_outputs -maxdepth 4 | sort | head -50
!for f in /kaggle/working/a4_outputs/logs/*.log; do echo "== $f"; grep -E "Score|Record F1|Dev set results|Error|error" "$f" | tail -5; done